In [1]:
# Libraries
import numpy as np
import os as os
import glob
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import netCDF4 as nc
import xarray as xr
import random as rd
import tqdm
import csv
plt.rcParams['figure.dpi'] = 300

In [12]:
### FUNCTION LIBRARY ###

def convert_to_datetime(year, month, day, time):
    """
    Convert year, month, day, and time to a numpy datetime64 object.

    Parameters
    ----------
    year : int
        Year.
    month : int
        Month.
    day : int
        Day.
    time : str
        Time in the format 'HH:MM:SS'.

    Returns
    -------
    np.datetime64
        Numpy datetime64 object.
    """
    date_str = f"{year}-{month}-{day}T{time}"
    return np.datetime64(date_str)

def process_GRUAN_filename(string):
    """
    Process GRUAN filename and extract site name and time information.

    Parameters
    ----------
    string : str
        GRUAN filename.

    Returns
    -------
    list
        List containing site name and time information.
    """
    split_string = string.split("_")
    site_name = split_string[0][:3]

    date_time_raw = split_string[4]
    year = date_time_raw[:4]
    month = date_time_raw[4:6]
    day = date_time_raw[6:8]

    time = date_time_raw[-6:]
    time = time[::-1]
    time = ":".join(time[i:i+2] for i in range(0, len(time), 2))
    time = time[::-1]

    time = convert_to_datetime(year, month, day, time)
    GRUAN_site_info = [site_name, time]
    return GRUAN_site_info

def match_files(GRUAN_date_sites, ERA5_file_paths):
    """
    Match GRUAN dates with ERA5 file names.

    Parameters
    ----------
    GRUAN_date_sites : list
        List of GRUAN dates and sites.
    ERA5_file_paths : list
        List of ERA5 file paths.

    Returns
    -------
    list
        List of matching ERA5 file names.
    """
    GRUAN_date_formatted = []
    ERA5_file_names = []
    ERA5_file_names_matching = []

    ERA5_file_names += [os.path.basename(path) for path in ERA5_file_paths]
    print(ERA5_file_names[0])

    for i in range(len(GRUAN_date_sites)):
        GRUAN_date_formatted =  str(GRUAN_date_sites[i][1]).replace("-", "_")[:10]
        GRUAN_date_formatted = GRUAN_date_formatted + '.nc'

        if GRUAN_date_formatted in ERA5_file_names:
            filename = [ERA5_file_names[ERA5_file_names.index(GRUAN_date_formatted)]]
            site_info = GRUAN_date_sites[i]
            ERA5_file_names_matching.append((filename.append(site_info[0])))
            ERA5_file_names_matching.append((filename.append(site_info[1])))
            print(ERA5_file_names_matching)
            print(site_info)


    return ERA5_file_names_matching

def press2alt(pressure):
    """
    Convert pressure to altitude.

    Parameters
    ----------
    pressure : Union[int, np.ndarray]
        Pressure in Pascal.

    Returns
    -------
    Union[float, np.ndarray]
        Altitude in meters.
    """
    L = -6.5*10**-3
    P0 = 101325
    T0 = 288.15
    R = 287.053
    g = 9.81

    altitudes = np.zeros_like(pressure)

    if type(pressure)==int:
        return (T0/L)*((pressure*100/P0)**(-R*L/g) -1)
    else:
        for i in range(len(pressure)):
            altitudes[i] = (T0/L)*((pressure[i]*100/P0)**(-R*L/g) -1)

        return altitudes

def compute_Psat_w(T):
    """
    Returns water liquid saturation pressure in Pascal.

    Parameters
    ----------
    T : Union[float, np.ndarray]
        Temperature in Kelvin.

    Returns
    -------
    Union[float, np.ndarray]
        H2O liquid saturation pressure in Pascal.
    """
    return 100.0 * np.exp(
        -6096.9385 / T
        + 16.635794
        - 0.02711193 * T
        + 1.673952e-5 * T**2
        + 2.433502 * np.log(T)
    )

def compute_Psat_i(T):
    """
    Returns water solid saturation pressure in Pascal.

    Parameters
    ----------
    T : Union[float, np.ndarray]
        Temperature in Kelvin.

    Returns
    -------
    Union[float, np.ndarray]
        H2O solid saturation pressure in Pascal.
    """
    return 100.0 * np.exp(
        -6024.5282 / T
        + 24.7219
        + 0.010613868 * T
        - 1.3198825e-5 * T**2
        - 0.49382577 * np.log(T)
    )

In [11]:
# This cell retrieves and processes data from ERA5 and GRUAN sources. 
# It creates a list of ERA5 file names, extracts the date and site information from GRUAN file names, and imports latitude and longitude locations of GRUAN sites. ...
# Finally, it picks out the ERA5 files that match the GRUAN dates.

# Make array of .nc filenames (grib)
ERA5_file_names = []
GRUAN_date_sites = []
years = np.linspace(2012,2016,num=9,dtype=int)

# Make list of all ERA5 file names
# Need to loop as glob is not capturing all file names
for i in range(len(years)):
    ERA5_file_names = ERA5_file_names + glob.glob('/home/chinahg/GCresearch/contrailuncertainty/ERA5_processing/ERA5_downloads/ERA5_downloads/'+str(years[i])+'/*.nc', recursive=True)
ERA5_num_files = len(ERA5_file_names)
ERA5_file_names.sort()

# Stripping GRUAN datetime and site data from file names so we can match with ERA5 data
GRUAN_file_names = [os.path.basename(file) for file in glob.glob('/home/chinahg/GCresearch/GRUAN_sondes/ftp.ncdc.noaa.gov/pub/data/gruan/processing/level2/RS92-GDP/version-002/' + '/**/*.nc', recursive=True)]
GRUAN_num_files = len(GRUAN_file_names)
GRUAN_file_names.sort()

for j in range(GRUAN_num_files):
    GRUAN_date_sites.append(process_GRUAN_filename(GRUAN_file_names[j]))

# Import lat-lon locations of GRUAN sites between 30-60 latitude
df_locations = pd.read_excel('/home/chinahg/GCresearch/contrailuncertainty/GRUAN_processing/GRUAN_site_data.xlsx', sheet_name='30-60lat')

# Create a list of ERA5 files that match the GRUAN dates
matching_files = match_files(GRUAN_date_sites, ERA5_file_names)
print(matching_files[0])


In [ ]:
# 10km is approx 265 hPa
print("Started!")

cruiseRH = []
MLD_array_pres = []
MLD_top = 0
MLD_bottom = 0
MLD_array_alt = []

T_t = 273.16 # Ice triple point temp [K]
P_ip = 6.12 # Ice triple point pressure [hPa]

press_upper = 250 #hPa
press_lower = 290 #hPa
alt_upper = press2alt(press_upper) # Upper altitude limit [m]
alt_lower = press2alt(press_lower) # Lower altitude limit [m]

# Make arrays of GRUAN site latitudes and longitudes
site_latitudes = df_locations['Latitude'].values
site_longitudes = df_locations['Longitude'].values

for k in tqdm.tqdm(range(matching_files)): # Look through all matching files
    for j in range(len(site_latitudes)): # We are looking at specific latitude and longitude sites (matching GRUAN sites)
        current_year = matching_files[k][0:4]
        path2file = "/home/chinahg/GCresearch/contrailuncertainty/ERA5_processing/ERA5_downloads/ERA5_downloads/"+current_year+"/"+matching_files[k]

        # Read file in the location of interest
        # View data for a single day
        try:
            ds_ERA5 = xr.open_dataset(path2file,engine='netcdf4')
        except: 
            break

        altitudes = press2alt(ds_ERA5.isobaricInhPa.to_numpy())

        # # Randomly sample the time index for the date chosen (so we don't have to process every single time/date)
        # ds_ERA5.t.sel(time=rand_time, latitude=site_latitudes[j], longitude=site_longitudes[j], method='nearest').to_numpy()
        # rand_time_idx = rd.randint(0,len(ds_ERA5.time)-1)
        # rand_time = ds_ERA5.time[rand_time_idx].to_numpy()

        # Assign ERA5 data to arrays
        RH_w = ds_ERA5.r.sel(time=rand_time, latitude=site_latitudes[j], longitude=site_longitudes[j], method='nearest').to_numpy() # [%] Water relative humidity
        T = ds_ERA5.t.sel(time=rand_time, latitude=site_latitudes[j], longitude=site_longitudes[j], method='nearest').to_numpy() # [K] Temperature

        # Calculate relative humidity wrt ice
        P_sat_w = compute_Psat_w(T) # [Pa] Bolton 1980
        P_sat_i =  compute_Psat_i(T) # [Pa] Guide to Meteorological Instruments and Methods of Observation (CIMO Guide) (WMO, 2008)
        RH_i = RH_w*P_sat_w/P_sat_i # [%]
        RH_len = len(RH_i)

        for i in range(RH_len): # look through all RH datapoints for all dates and locations

            if altitudes[i] >= alt_lower and altitudes[i] <= alt_upper and RH_i[i] >= 100: # if pressure is approx 265 hPa and RH > 100% record RH
                # Save supersaturated RH value
                cruiseRH.append(RH_i[i])

                # Save altitude where supersaturated RH starts
                MLD_top = altitudes[i]

                for j in range(i): # Look through list of RH under cruise alt and determine MLD

                    if RH_i[i-j] < 100: # MLD ends when RHi < 100%
                        MLD_index = i-j
                        MLD_bottom = altitudes[MLD_index]
                        break
                    elif j == i:
                        MLD_bottom = altitudes[0]
                        break
                # Now have an array of RH and MLD upper and lower bounds
                # Take difference of altitudes to get MLD in [m]
                MLD_array_alt.append(MLD_top-MLD_bottom)
        ds_ERA5.close()


        # MLD_array_alt is appended to for each file, never overwritten
        # cruiseRH is appended to for each file, never overwritten

In [ ]:
# Save the data so we don't have to process it again
# Specify the file path
MLD_file_path = 'ERA5_MLD.csv'
RH_file_path = 'ERA5_RH.csv'

# Open the file in write mode
with open(MLD_file_path, 'w', newline='') as csvfile:
    # Create a CSV writer object
    writer = csv.writer(csvfile)
    
    # Write the array to the CSV file
    writer.writerow(MLD_array_alt)

with open(RH_file_path, 'w', newline='') as csvfile:
    # Create a CSV writer object
    writer = csv.writer(csvfile)
    
    # Write the array to the CSV file
    writer.writerow(cruiseRH)

In [ ]:
ds_ERA5 = xr.open_dataset(ERA5_file_names[0],engine='netcdf4')


In [ ]:


process_string("BAR-RS-01_2_RS92-GDP_002_20090101T060000_1-000-001.nc")[0]